# CURE-Rec — end-to-end quickstart

This notebook runs the complete first implementation milestone: configuration and data loading, CURE-Sim creation, exact 64-coalition evaluation, Shapley/interaction regions, robust improvement selection, and structured-log inspection.

**Important:** CURE-Sim is an oracle benchmark. The optional CSV audit below intentionally does not turn an arbitrary interaction file into causal evidence.

## 1. Setup

Run from `paper-ideas/CURE-Rec/code/` after installing `pip install -e '.[dev]'`. The cell also makes the notebook usable directly from a source checkout.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cure_rec.config import load_settings
from cure_rec.data import audit_interactions, load_curesim, load_interactions_csv
from cure_rec.pipeline import run_experiment

print('Project root:', ROOT)

## 2. Load the experiment configuration and synthetic data-generating environment

`CURE-Sim` is the primary data source for this milestone because it exposes known policy effects, feedback dynamics, and oracle coalition values. The configuration is part of the run manifest.

In [ ]:
config_path = ROOT / 'configs' / 'curesim_quickstart.yaml'
settings = load_settings(config_path)
print('Config hash:', settings.config_hash())
print('Users / items / horizon:', settings.simulator.n_users, settings.simulator.n_items, settings.simulator.horizon)
print('Interventions:', list(settings.interventions.costs))
print('Scenarios:', [scenario.name for scenario in settings.scenarios])
# Explicit synthetic data loading: inspect the disclosed starting state.
simulator = load_curesim(settings)
print('Synthetic catalogue shape:', simulator.catalog.features.shape)
print('Synthetic public-profile shape:', simulator.state.public_profiles.shape)


## 3. Run the exact intervention game

This evaluates all 64 intervention coalitions under every configured scenario. The output includes:

- full-game Shapley regions;
- feasibility-aware semivalue sensitivity;
- Grabisch–Roubens interaction regions;
- direct robust improvement selection with abstention;
- JSONL event logs and CSV artifacts.

In [ ]:
logger, game, decision = run_experiment(settings)
RUN_DIR = logger.run_dir
print('Run directory:', RUN_DIR)
print('Decision:', decision.action)
print('Selected portfolio:', decision.selected_interventions)
print('Worst-case improvement:', round(decision.lower_improvement, 5))

## 4. Inspect exact attribution and interaction outputs

In [ ]:
display(game.regions.sort_values('phi_mean', ascending=False))
display(game.interaction_table.sort_values('interaction_mean', ascending=False))

## 5. Inspect the direct robust portfolio table

The planner selects by scenario-wise worst-case improvement, not by adding lower Shapley endpoints.

In [ ]:
coalitions = game.coalition_table.groupby('mask', as_index=False).agg(
    lower_improvement=('improvement', 'min'),
    upper_improvement=('improvement', 'max'),
    cost=('cost', 'first'),
    interventions=('active_interventions', 'first'),
).sort_values('lower_improvement', ascending=False)
display(coalitions.head(12))

## 6. Inspect structured logs and artifacts

The event stream is JSONL, so it is inspectable with Pandas, `jq`, or a text editor. Per-coalition records contain metrics, timing, active interventions, and transform statistics.

In [ ]:
import json
import pandas as pd

events_path = RUN_DIR / 'logs' / 'events.jsonl'
events = pd.DataFrame([json.loads(line) for line in events_path.read_text().splitlines()])
display(events[['timestamp_utc', 'event']].tail(12))

for artifact in sorted((RUN_DIR / 'artifacts').glob('*.json')):
    print('artifact:', artifact.name)
for figure in sorted((RUN_DIR / 'figures').glob('*.png')):
    print('figure:', figure.name)

## 7. Optional: audit a local interaction CSV

Set `LOCAL_CSV` to a local file containing at least `user_id`, `item_id`, `timestamp`, and `response`. The audit labels the strongest claim supported by the columns; it does not infer missing propensities or exposure fields.

In [ ]:
LOCAL_CSV = None  # e.g. ROOT / 'data' / 'raw' / 'interactions.csv'
if LOCAL_CSV is not None:
    frame = load_interactions_csv(LOCAL_CSV)
    audit = audit_interactions(frame)
    print(audit)
else:
    print('No local CSV selected; synthetic CURE-Sim run above is complete.')

## 8. Next run

After the quickstart passes, switch to `configs/curesim_full.yaml`, inspect `runs/<run-id>/artifacts/explanation_card.json`, and only then begin audited real-log integration.